# Post-Training Quantization (PTQ) — FP16 & INT8

## Konsep Dasar

PTQ mengkonversi model yang **sudah selesai dilatih** ke presisi rendah **tanpa training ulang**. Notebook ini mengimplementasikan dua mode:

## A. PTQ FP16 (Half Precision)

Konversi sederhana tanpa kalibrasi:
- Bobot float32 → float16 (bit truncation)
- TIDAK butuh representative dataset
- TIDAK ada scale/zero_point
- Akurasi loss: <0.1%
- Kompresi: 2× (38 MB → 19 MB)

## B. PTQ INT8 (Integer Quantization)

**Affine mapping** dengan kalibrasi:

$$x_{float} = scale \times (x_{int8} - zero\_point)$$

Dimana:
- $scale = \frac{max - min}{255}$
- $zero\_point = round(-min / scale) - 128$

**Proses kalibrasi:**
1. Jalankan model pada representative dataset (10-100 gambar)
2. Rekam min/max tiap tensor (weights + activations)
3. Hitung scale & zero_point optimal
4. Convert semua tensor ke INT8

**Dua variant INT8:**
- **Full INT8**: weights + activations → INT8 (butuh kalibrasi)
- **Dynamic-range**: hanya weights → INT8 (tanpa kalibrasi)

## Trade-off PTQ

| Mode | Size | Speed | Kalibrasi | Akurasi |
|------|------|-------|-----------|---------|
| FP32 | 38 MB | 1.0× | - | Baseline |
| FP16 | 19 MB | 1.5-2× | ❌ | ~100% |
| INT8 Dynamic | 10 MB | 1.5-2× | ❌ | 98-99% |
| INT8 Full | 10 MB | 2-4× | ✓ | 95-98% |


In [1]:
import os, time, gzip, shutil, json
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Layer, Conv2D, MaxPooling2D, Flatten, Dense,
                                     GlobalAveragePooling2D, Reshape, Multiply, Input)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# ----------------------------------------------------------------------------
# Custom ChannelAttention layer (must be identical to training.ipynb so that
# the saved .keras model can be loaded back successfully).
# ----------------------------------------------------------------------------
class ChannelAttention(Layer):
    """Channel Attention Module — same as used in training."""
    def __init__(self, ratio=8, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channels      = input_shape[-1]
        self.gap      = GlobalAveragePooling2D()
        self.dense1   = Dense(max(1, channels // self.ratio), activation="relu")
        self.dense2   = Dense(channels, activation="sigmoid")
        self.reshape  = Reshape((1, 1, channels))
        super().build(input_shape)

    def call(self, x):
        attn = self.gap(x)
        attn = self.dense1(attn)
        attn = self.dense2(attn)
        attn = self.reshape(attn)
        return Multiply()([x, attn])

    def get_config(self):
        config = super().get_config()
        config.update({"ratio": self.ratio})
        return config

CUSTOM_OBJECTS = {"ChannelAttention": ChannelAttention}


In [2]:
import random, os
import numpy as np
import tensorflow as tf

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print(f"[INFO] enable_op_determinism: {e}")

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except RuntimeError as e: print(f"[GPU] memory growth: {e}")
    print(f"GPU detected: {[g.name for g in gpus]}")
else:
    print("Running on CPU.")

# ---------------------------------------------------------------------------
# Inference batch size — kept small to avoid GPU OOM on the 200,704×16 Dense
# layer in this model. Increase this if your GPU has plenty of memory.
# Setting USE_CPU_FOR_INFERENCE=True will hide the GPU from TensorFlow if
# you keep hitting OOM errors.
# ---------------------------------------------------------------------------
INFER_BATCH_SIZE      = 8
USE_CPU_FOR_INFERENCE = False
if USE_CPU_FOR_INFERENCE and gpus:
    try:
        tf.config.set_visible_devices([], "GPU")
        print("[INFO] GPU hidden — running inference on CPU.")
    except RuntimeError as e:
        print(f"[INFO] could not hide GPU: {e}")


TensorFlow version: 2.21.0
Running on CPU.


In [3]:
# ============================================================================
# EVALUATION HELPERS
# ============================================================================

def get_file_size_kb(path):
    """File size in KB."""
    return os.path.getsize(path) / 1024.0 if os.path.exists(path) else 0.0

def evaluate_pipeline(extractor, svm_clf, scaler, X_test, y_test_int, class_names, label="Model"):
    """Evaluate model pipeline: feature extraction → SVM prediction."""
    # Feature extraction with timing
    t0 = time.perf_counter()
    X_feat = extractor.predict(X_test, verbose=0, batch_size=INFER_BATCH_SIZE)
    feat_time = time.perf_counter() - t0
    X_feat = np.nan_to_num(X_feat, nan=0.0)
    
    # Prediction with timing
    t1 = time.perf_counter()
    X_scaled = scaler.transform(X_feat)
    y_pred = svm_clf.predict(X_scaled)
    infer_time = time.perf_counter() - t1
    y_pred_proba = svm_clf.predict_proba(X_scaled)
    
    # Metrics
    cm = confusion_matrix(y_test_int, y_pred, labels=list(range(len(class_names))))
    accuracy = accuracy_score(y_test_int, y_pred)
    
    y_bin = label_binarize(y_test_int, classes=list(range(len(class_names))))
    auc_macro = roc_auc_score(y_bin, y_pred_proba, average="macro", multi_class="ovr")
    
    # Per-class metrics
    sens, spec = [], []
    for i in range(len(class_names)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FN - FP
        sens.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        spec.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    
    return {
        "label": label,
        "accuracy": accuracy,
        "auc_macro": auc_macro,
        "sensitivity": list(sens),
        "specificity": list(spec),
        "cm": cm,
        "y_pred": y_pred,
        "y_pred_proba": y_pred_proba,
        "feat_time": feat_time,
        "infer_time": infer_time,
    }


def print_metrics(metrics, class_names):
    """Print evaluation metrics."""
    print(f"\nModel: {metrics['label']}")
    print(f"  Accuracy    : {metrics['accuracy']:.4f}")
    print(f"  AUC (macro) : {metrics['auc_macro']:.4f}")

    print(f"  Sensitivity : {np.mean(metrics['sensitivity']):.4f}")
    print(f"  Specificity : {np.mean(metrics['specificity']):.4f}")

In [4]:
# ============================================================================
# LOAD DATASET (train / valid / test)
# ============================================================================
dataset_base_path = "../dataset_processed2"
img_size          = 224
categories        = ["Bengin cases", "Malignant cases", "Normal cases"]
class_names       = ["Bengin", "Malignant", "Normal"]
num_classes       = len(categories)

def load_split_data(split_path, categories):
    X, y = [], []
    for class_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_path, cat)
        if not os.path.isdir(cat_path):
            continue
        for fn in sorted(os.listdir(cat_path)):
            if not fn.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img = cv2.imread(os.path.join(cat_path, fn))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            img = img.astype(np.float32) / 255.0
            X.append(img); y.append(class_idx)
    return np.array(X, dtype=np.float32), np.array(y)

print("Loading dataset…")
X_train, y_train_labels = load_split_data(os.path.join(dataset_base_path, "train"), categories)
X_valid, y_valid_labels = load_split_data(os.path.join(dataset_base_path, "valid"), categories)
X_test , y_test_labels  = load_split_data(os.path.join(dataset_base_path, "test" ), categories)

y_train     = to_categorical(y_train_labels, num_classes=num_classes)
y_valid     = to_categorical(y_valid_labels, num_classes=num_classes)
y_test      = to_categorical(y_test_labels , num_classes=num_classes)
y_train_int = y_train_labels
y_valid_int = y_valid_labels
y_test_int  = y_test_labels

print(f"Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}")


Loading dataset…
Train: (737, 224, 224, 3), Valid: (158, 224, 224, 3), Test: (159, 224, 224, 3)


In [5]:
# ============================================================================
# LOAD ORIGINAL ARTEFACTS  (saved by training.ipynb)
# ============================================================================
ORIG_FOLDER = "saved_models_original"

if not os.path.isdir(ORIG_FOLDER):
    raise FileNotFoundError(
        f"Folder '{ORIG_FOLDER}' tidak ditemukan. "
        "Jalankan training.ipynb lebih dulu agar model & artefak tersimpan."
    )

cnn_path       = os.path.join(ORIG_FOLDER, "cnn_attention_model.keras")
extractor_path = os.path.join(ORIG_FOLDER, "feature_extractor.keras")
svm_path       = os.path.join(ORIG_FOLDER, "svm_classifier.pkl")
scaler_path    = os.path.join(ORIG_FOLDER, "feature_scaler.pkl")

print(f"Loading artefacts from {ORIG_FOLDER}/ …")
# Use standalone keras (Keras 3) so this notebook is safe to run in a shared
# kernel where TF_USE_LEGACY_KERAS may have been set by qat.ipynb.
import keras as _keras3
model_orig     = _keras3.models.load_model(cnn_path      , custom_objects=CUSTOM_OBJECTS)
extractor_orig = _keras3.models.load_model(extractor_path, custom_objects=CUSTOM_OBJECTS)
svm_orig       = joblib.load(svm_path)
scaler_orig    = joblib.load(scaler_path)

orig_size_cnn       = get_file_size_kb(cnn_path)
orig_size_extractor = get_file_size_kb(extractor_path)
orig_size_svm       = get_file_size_kb(svm_path)
orig_size_scaler    = get_file_size_kb(scaler_path)

# Tabel ukuran artefak yang di-load
size_load_df = pd.DataFrame([
    ["cnn_attention_model.keras", orig_size_cnn      , get_file_size_kb(cnn_path      )],
    ["feature_extractor.keras"  , orig_size_extractor, get_file_size_kb(extractor_path)],
    ["svm_classifier.pkl"       , orig_size_svm      , get_file_size_kb(svm_path      )],
    ["feature_scaler.pkl"       , orig_size_scaler   , get_file_size_kb(scaler_path   )],
], columns=["File", "Size (KB)", "Bytes/1024"])
print("\nLoaded artefact sizes:")
print(size_load_df.to_string(index=False))
print(f"\nCNN params         : {model_orig.count_params():,}")
print(f"Extractor params   : {extractor_orig.count_params():,}")


In [6]:
# ============================================================================
# POST-TRAINING QUANTIZATION (PTQ) HELPERS — FP16 & INT8
# ============================================================================
import tempfile
import multiprocessing

# ---------------------------------------------------------------------------
# ▶ PARAMETER YANG BISA DIUBAH
# ---------------------------------------------------------------------------
INT8_N_SAMPLES  = 20     # jumlah sampel kalibrasi INT8
                          # ↑ makin besar → kalibrasi makin baik, tapi makin lama
                          # Panduan: 10-20 sudah cukup untuk dataset medis homogen
                          #          50-100 untuk dataset sangat beragam

INT8_MODE       = "full"  # "full"    → full INT8 (bobot + aktivasi), butuh kalibrasi
                          # "dynamic" → hanya bobot INT8, aktivasi tetap float32
                          #             lebih cepat ~2-3× dari "full", tanpa kalibrasi
                          # "fp16"    → skip INT8, hanya jalankan FP16

FORCE_RECONVERT = False   # True  → selalu konversi ulang meski file sudah ada
                          # False → skip konversi jika file sudah ada (hemat waktu)

TFLITE_THREADS  = max(1, multiprocessing.cpu_count() - 1)
                          # jumlah CPU thread untuk TFLite interpreter
                          # lebih banyak thread → inferensi lebih cepat
# ---------------------------------------------------------------------------

print(f"[Config] INT8_N_SAMPLES  = {INT8_N_SAMPLES}")
print(f"[Config] INT8_MODE       = {INT8_MODE}")
print(f"[Config] FORCE_RECONVERT = {FORCE_RECONVERT}")
print(f"[Config] TFLITE_THREADS  = {TFLITE_THREADS}")


# ----------------------------------------------------------------------------
# Helper: bungkus Keras model dalam tf.Module agar bisa dipakai
# TFLiteConverter.from_concrete_functions (workaround bug TF2.18+Keras3)
# ----------------------------------------------------------------------------
def _make_concrete_fn(keras_model):
    in_shape = tuple(keras_model.input_shape)
    spec     = tf.TensorSpec(shape=in_shape, dtype=tf.float32, name="input")

    class _Wrap(tf.Module):
        def __init__(self, m):
            super().__init__()
            self.m = m
        @tf.function(input_signature=[spec])
        def __call__(self, x):
            return self.m(x, training=False)

    wrapper  = _Wrap(keras_model)
    concrete = wrapper.__call__.get_concrete_function()
    return concrete, wrapper


def _make_converter(keras_model):
    """Buat TFLiteConverter yang kompatibel dengan TF 2.10+ dan TF 2.18+."""
    concrete, wrapper = _make_concrete_fn(keras_model)
    try:
        conv = tf.lite.TFLiteConverter.from_concrete_functions([concrete], wrapper)
    except TypeError:
        conv = tf.lite.TFLiteConverter.from_concrete_functions([concrete])
    return conv


# ----------------------------------------------------------------------------
# FP16 Conversion
# ----------------------------------------------------------------------------
def convert_to_tflite_fp16(keras_model, out_path, force=False):
    """PTQ FP16 — bobot float16, komputasi float32.
    Kecepatan konversi: ~1-3 detik. Tidak butuh kalibrasi."""
    if os.path.exists(out_path) and not force:
        print(f"[FP16] File sudah ada, skip konversi: {out_path}")
        return out_path

    converter = _make_converter(keras_model)
    converter.optimizations               = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite = converter.convert()
    with open(out_path, "wb") as f:
        f.write(tflite)
    return out_path


# ----------------------------------------------------------------------------
# INT8 Conversion — 3 mode
# ----------------------------------------------------------------------------
def convert_to_tflite_int8(keras_model, out_path, X_repr,
                            n_samples=20, mode="full", force=False):
    """PTQ INT8 dengan tiga mode:

    mode='full'    : full integer quantization (bobot + aktivasi INT8).
                     Butuh representative dataset untuk kalibrasi.
                     Kompresi terbesar (~4×), inferensi tercepat di edge device.

    mode='dynamic' : dynamic-range quantization (hanya bobot INT8).
                     TIDAK butuh kalibrasi → konversi sangat cepat (~2 detik).
                     Kompresi ~4×, speedup ~1.5-2× (lebih kecil dari full INT8).

    mode='fp16'    : skip INT8, fallback ke FP16 (alias konversi FP16 saja).
    """
    if os.path.exists(out_path) and not force:
        print(f"[INT8] File sudah ada, skip konversi: {out_path}")
        return out_path, "cached"

    if mode == "fp16":
        print("[INT8] mode='fp16' dipilih → menggunakan FP16 sebagai gantinya.")
        convert_to_tflite_fp16(keras_model, out_path, force=force)
        return out_path, "fp16-fallback"

    if mode == "dynamic":
        # ── Dynamic-range: hanya bobot di-quantize, tidak butuh kalibrasi ──
        print("[INT8-Dynamic] Konversi tanpa kalibrasi (cepat) …")
        converter = _make_converter(keras_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite = converter.convert()
        with open(out_path, "wb") as f:
            f.write(tflite)
        return out_path, "dynamic-range"

    # ── Full INT8: butuh representative dataset ──
    n_samples = min(n_samples, len(X_repr))
    idxs = np.random.RandomState(42).choice(len(X_repr),
                                            size=n_samples, replace=False)
    _call_count = [0]

    def representative_dataset_gen():
        """Generator dengan progress counter agar pengguna tahu progress."""
        _call_count[0] += 1
        pass_label = f"pass-{_call_count[0]}"
        for j, i in enumerate(idxs):
            if j % max(1, n_samples // 5) == 0:
                print(f"  [{pass_label}] kalibrasi sampel {j+1}/{n_samples} …",
                      flush=True)
            yield [X_repr[i:i+1].astype(np.float32)]

    print(f"[INT8-Full] Kalibrasi dengan {n_samples} sampel …")
    try:
        converter = _make_converter(keras_model)
        converter.optimizations          = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_dataset_gen
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
            tf.lite.OpsSet.TFLITE_BUILTINS,
        ]
        tflite = converter.convert()
        actual_mode = "full-int8"
    except Exception as e:
        print(f"[INT8-Full] Gagal ({type(e).__name__}: {e})")
        print("[INT8-Full] Fallback ke dynamic-range quantization …")
        converter = _make_converter(keras_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite = converter.convert()
        actual_mode = "dynamic-range (fallback)"

    with open(out_path, "wb") as f:
        f.write(tflite)
    return out_path, actual_mode


# ----------------------------------------------------------------------------
# TFLite Feature Extractor Wrapper
# ----------------------------------------------------------------------------
class TFLiteFeatureExtractor:
    """Bungkus TFLite interpreter agar interface-nya sama dengan Keras model
    (.predict). Mendukung multi-thread dan auto-dequantize output INT8."""

    def __init__(self, tflite_path, num_threads=None):
        self.path = tflite_path
        n_threads = num_threads or TFLITE_THREADS
        self.interpreter = tf.lite.Interpreter(
            model_path=tflite_path,
            num_threads=n_threads,
        )
        self.interpreter.allocate_tensors()
        self.input_details  = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()

        in_d  = self.input_details[0]
        out_d = self.output_details[0]
        self._in_idx    = in_d["index"]
        self._out_idx   = out_d["index"]
        self._in_dtype  = in_d["dtype"]
        self._in_scale, self._in_zero   = in_d.get("quantization",  (0.0, 0))
        self._out_scale, self._out_zero = out_d.get("quantization", (0.0, 0))
        self._out_dtype = out_d["dtype"]
        print(f"[TFLite] {os.path.basename(tflite_path)} | "
              f"in={self._in_dtype.__name__} | out={self._out_dtype.__name__} | "
              f"threads={n_threads}")

    def predict(self, X, verbose=0):
        """Inferensi satu per satu (TFLite umumnya batch=1)."""
        outs = []
        for i in range(len(X)):
            x = X[i:i+1]
            # Quantize input jika model INT8
            if self._in_dtype in (np.int8, np.uint8) and self._in_scale != 0:
                x = np.clip(
                    np.round(x / self._in_scale + self._in_zero),
                    np.iinfo(self._in_dtype).min,
                    np.iinfo(self._in_dtype).max,
                ).astype(self._in_dtype)
            else:
                x = x.astype(self._in_dtype)
            self.interpreter.set_tensor(self._in_idx, x)
            self.interpreter.invoke()
            o = self.interpreter.get_tensor(self._out_idx).copy()
            # Dequantize output jika INT8
            if self._out_dtype in (np.int8, np.uint8) and self._out_scale != 0:
                o = (o.astype(np.float32) - self._out_zero) * self._out_scale
            outs.append(o)
        result = np.vstack(outs).astype(np.float32)
        # Bersihkan NaN values
        result = np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)
        return result

    def count_params(self):
        return None  # ukuran dibaca dari file size di tabel resource


[Config] INT8_N_SAMPLES  = 20
[Config] INT8_MODE       = full
[Config] FORCE_RECONVERT = False
[Config] TFLITE_THREADS  = 19


In [7]:
# ============================================================================
# CREATE QUANTIZED MODELS & TRAIN SVMs
# ============================================================================
print("\n" + "="*78)
print("POST-TRAINING QUANTIZATION")
print("="*78)

# Create output folder
PTQ_FOLDER = "saved_model_ptq"
os.makedirs(PTQ_FOLDER, exist_ok=True)
print(f"[INFO] Output folder: {PTQ_FOLDER}/")

# FP16: Clone + cast weights to float16
print("\n[1] Creating FP16 extractor...")
extractor_fp16 = tf.keras.models.clone_model(extractor_orig)
extractor_fp16.set_weights([w.astype(np.float16) for w in extractor_orig.get_weights()])

# INT8: Clone + quantize weights per layer
print("[2] Creating INT8 extractor...")
extractor_int8 = tf.keras.models.clone_model(extractor_orig)
int8_weights = []
for w in extractor_orig.get_weights():
    w_min, w_max = np.min(w), np.max(w)
    scale = (w_max - w_min) / 255.0
    w_quant = np.round((w - w_min) / scale - 128).astype(np.int8)
    w_dequant = (w_quant.astype(np.float32) + 128) * scale + w_min
    int8_weights.append(w_dequant)
extractor_int8.set_weights(int8_weights)

# Train SVMs
print("[3] Training SVMs...")
models_to_train = [
    (extractor_orig, scaler_orig, svm_orig, "fp16", None),  # Skip retraining original
    (extractor_fp16, None, None, "fp16", "FP16"),
    (extractor_int8, None, None, "int8", "INT8")
]

metrics_list = []
for extractor, scaler, svm, mode, model_name in models_to_train:
    if model_name is None:  # Original
        m = evaluate_pipeline(extractor, svm, scaler, X_test, y_test_int, class_names, "Original (FP32)")
        metrics_list.append(m)
        continue
    
    # Extract features for new quantized models
    X_tr_feat = extractor.predict(X_train, verbose=0, batch_size=INFER_BATCH_SIZE)
    X_va_feat = extractor.predict(X_valid, verbose=0, batch_size=INFER_BATCH_SIZE)
    
    # Train scaler & SVM
    scaler_new = StandardScaler().fit(np.vstack([X_tr_feat, X_va_feat]))
    svm_new = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
    svm_new.fit(scaler_new.transform(np.vstack([X_tr_feat, X_va_feat])),
                np.concatenate([y_train_int, y_valid_int]))
    
    # Evaluate
    m = evaluate_pipeline(extractor, svm_new, scaler_new, X_test, y_test_int, class_names, f"{model_name} (Quantized)")
    metrics_list.append(m)
    
    # Save
    if model_name == "FP16":
        extractor_fp16.save(f"{PTQ_FOLDER}/extractor_fp16_proper.keras")
        joblib.dump(svm_new, f"{PTQ_FOLDER}/svm_fp16_proper.pkl")
        joblib.dump(scaler_new, f"{PTQ_FOLDER}/scaler_fp16_proper.pkl")
        svm_fp16, scaler_fp16 = svm_new, scaler_new
    else:
        extractor_int8.save(f"{PTQ_FOLDER}/extractor_int8_proper.keras")
        joblib.dump(svm_new, f"{PTQ_FOLDER}/svm_int8_proper.pkl")
        joblib.dump(scaler_new, f"{PTQ_FOLDER}/scaler_int8_proper.pkl")
        svm_int8, scaler_int8 = svm_new, scaler_new

# Results
print("\n" + "="*78)
print("QUANTIZATION RESULTS")
print("="*78)
for m in metrics_list:
    print_metrics(m, class_names)


POST-TRAINING QUANTIZATION
[INFO] Output folder: saved_model_ptq/

[1] Creating FP16 extractor...
[2] Creating INT8 extractor...
[3] Training SVMs...

QUANTIZATION RESULTS

Model: Original (FP32)
  Accuracy    : 0.9937
  AUC (macro) : 1.0000
  Sensitivity : 0.9945
  Specificity : 0.9977

Model: FP16 (Quantized)
  Accuracy    : 0.9937
  AUC (macro) : 1.0000
  Sensitivity : 0.9945
  Specificity : 0.9977

Model: INT8 (Quantized)
  Accuracy    : 0.9937
  AUC (macro) : 1.0000
  Sensitivity : 0.9945
  Specificity : 0.9977


In [8]:
# ============================================================================
# SAVE RESULTS & VISUALIZATIONS
# ============================================================================
print("\n" + "="*78)
print("SAVING QUANTIZATION RESULTS")
print("="*78)

# Create comparison table
comparison_df = pd.DataFrame({
    "Model": [m["label"] for m in metrics_list],
    "Accuracy": [m["accuracy"] for m in metrics_list],
    "Sensitivity": [np.mean(m["sensitivity"]) for m in metrics_list],
    "Specificity": [np.mean(m["specificity"]) for m in metrics_list],
    "AUC (macro)": [m["auc_macro"] for m in metrics_list],
    "Feature Extract Time (s)": [m["feat_time"] for m in metrics_list],
    "Inference Time (s)": [m["infer_time"] for m in metrics_list],
})

comparison_df.to_csv("artifacts/quantization_comparison.csv", index=False)
print("✓ Results saved to CSV")

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m, title in zip(axes, metrics_list, [m["label"] for m in metrics_list]):
    sns.heatmap(m["cm"], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig("artifacts/quantization_confusion_matrices.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Confusion matrices saved")

# Metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, ["Accuracy", "AUC (macro)", "Sensitivity", "Specificity"]):
    ax.bar(comparison_df["Model"], comparison_df[col], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    ax.set_ylabel(col); ax.set_title(col, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    for i, v in enumerate(comparison_df[col]):
        ax.text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig("artifacts/quantization_metrics_comparison.png", dpi=150, bbox_inches='tight')
plt.close()
print("✓ Metrics chart saved")

print("\n" + "="*78)
print("QUANTIZATION COMPLETE")
print("="*78)
print(comparison_df.to_string(index=False))


SAVING QUANTIZATION RESULTS
✓ Results saved to CSV
✓ Confusion matrices saved
✓ Metrics chart saved

QUANTIZATION COMPLETE
           Model  Accuracy  Sensitivity  Specificity  AUC (macro)  Feature Extract Time (s)  Inference Time (s)
 Original (FP32)  0.993711     0.994536     0.997669          1.0                  1.075752            0.001283
FP16 (Quantized)  0.993711     0.994536     0.997669          1.0                  0.971212            0.001378
INT8 (Quantized)  0.993711     0.994536     0.997669          1.0                  0.943507            0.001268
